# Daily Challenge: Build a Tiny Agent with Tools (smolagents) - 👩‍🏫 TEACHER (solution)

**Goal:** a beginner agent that calls simple tools with the **smolagents** framework, using no paid APIs and a tiny local model or a stub.

```
question ─► ToolCallingAgent ─► model proposes a tool call ─► kb_lookup / math ─► observation ─► ... ─► final_answer
```
1. A small knowledge base (`kb_snippets`, 6 snippets with sources).
2. Two tools: `KBLookupTool` (keyword search with `[kb:N]` tags) and `MathTool` (add or multiply).
3. A model:
   - **stub (default):** a deterministic smolagents `Model` that emits real tool calls;
   - `TransformersModel("sshleifer/tiny-gpt2")`, as the exercise asks;
   - optionally, a small instruction-tuned model.
4. A `ToolCallingAgent`.
5. Three test questions, with the tool calls and final answers inspected.

In [1]:
!pip install -q "smolagents[transformers]" wikipedia

In [2]:
import os, re, json
os.environ["USE_TF"] = "0"
from importlib.metadata import version
import smolagents
from smolagents import Tool, ToolCallingAgent, TransformersModel
from smolagents.models import Model, ChatMessage, ChatMessageToolCall, ChatMessageToolCallFunction, MessageRole
print("smolagents", version("smolagents"), "| transformers", version("transformers"))

smolagents 1.26.0 | transformers 4.57.6


## 1. Knowledge base

In [3]:
kb_snippets = [
    {"id": 1, "source": "course-notes/agents", "text": "An agentic AI loop repeats observe -> think/plan -> act (call a tool) -> observe the result, until the goal is reached or a step limit stops it."},
    {"id": 2, "source": "course-notes/agents", "text": "Tools give an agent abilities beyond text generation, such as search, calculators, code execution or APIs; each tool has a name, a description and typed inputs."},
    {"id": 3, "source": "smolagents-docs", "text": "smolagents provides two agent types: CodeAgent writes its actions as Python code, while ToolCallingAgent emits JSON tool calls."},
    {"id": 4, "source": "course-notes/rag", "text": "Retrieval-augmented generation grounds answers in retrieved documents and should cite the sources it used."},
    {"id": 5, "source": "course-notes/safety", "text": "Agents should cap the number of steps, validate tool arguments and say when evidence is missing instead of guessing."},
    {"id": 6, "source": "course-notes/models", "text": "Tiny models such as sshleifer/tiny-gpt2 are useful for testing pipelines but cannot reliably follow instructions or produce tool calls."},
]
print(len(kb_snippets), "snippets")

6 snippets


## 2. Tools
A smolagents tool subclasses `Tool` and declares a `name`, a `description`, typed `inputs` and an `output_type`. The agent shows this schema to the model.

In [4]:
STOPWORDS = set("a an the is are what how why who of to in on for and or with by an do does".split())

class KBLookupTool(Tool):
    name = "kb_lookup"
    description = ("Look up the course knowledge base. Returns the snippets containing the query keywords, "
                   "each prefixed with its citation tag like [kb:1]. Returns 'NO_MATCH' if nothing matches.")
    inputs = {"query": {"type": "string", "description": "Keywords or a question to search for."}}
    output_type = "string"

    def forward(self, query: str) -> str:
        words = [w for w in re.findall(r"[a-z]+", query.lower()) if w not in STOPWORDS and len(w) > 2]
        scored = []
        for s in kb_snippets:
            text = s["text"].lower()
            score = sum(w in text or w.rstrip("s") in text for w in words)
            if score:
                scored.append((score, s))
        scored.sort(key=lambda x: -x[0])
        if not scored:
            return "NO_MATCH"
        return "\n".join(f"[kb:{s['id']}] ({s['source']}) {s['text']}" for _, s in scored[:3])


class MathTool(Tool):
    name = "math"
    description = "Add or multiply two numbers. Use op='add' or op='multiply'."
    inputs = {
        "a": {"type": "number", "description": "First number."},
        "b": {"type": "number", "description": "Second number."},
        "op": {"type": "string", "description": "Either 'add' or 'multiply'."},
    }
    output_type = "number"

    def forward(self, a: float, b: float, op: str) -> float:
        op = op.lower().strip()
        if op in ("add", "+", "sum", "plus"):
            r = a + b
        elif op in ("multiply", "*", "x", "times", "product"):
            r = a * b
        else:
            raise ValueError(f"Unsupported op {op!r}; use 'add' or 'multiply'.")
        return int(r) if float(r).is_integer() else r


kb_tool, math_tool = KBLookupTool(), MathTool()
print(kb_tool("agentic loop")); print(math_tool(12, 30, "add"), math_tool(7, 6, "multiply")); print(kb_tool("quantum bananas"))

[kb:1] (course-notes/agents) An agentic AI loop repeats observe -> think/plan -> act (call a tool) -> observe the result, until the goal is reached or a step limit stops it.
42 42
NO_MATCH


## 3. The model
- **`StubToolModel` (default):** a tiny smolagents `Model` that behaves like a tool-calling LLM.
  - With no observation yet, it chooses `math` (if the question contains numbers plus add/multiply) or `kb_lookup`.
  - Once the tool has answered, it calls `final_answer` with a 2–4 sentence answer: the KB citations are kept, and if the KB returned `NO_MATCH` it says so and proposes a follow-up question.
- **`TransformersModel("sshleifer/tiny-gpt2")`:** the exercise's default "real" model. It is a 2-layer, randomly initialised test model, so it cannot produce valid tool-call JSON, as we show below.

In [5]:
def _text(msg):
    c = msg.content
    return c if isinstance(c, str) else " ".join(x.get("text", "") for x in (c or []) if isinstance(x, dict))

class StubToolModel(Model):
    """Deterministic stand-in for a tool-calling LLM (no weights, no API key)."""

    def _call(self, name, args):
        self.n = getattr(self, "n", 0) + 1
        return ChatMessage(role=MessageRole.ASSISTANT, content="",
                           tool_calls=[ChatMessageToolCall(id=f"call_{self.n}", type="function",
                                                           function=ChatMessageToolCallFunction(name=name, arguments=args))])

    def generate(self, messages, stop_sequences=None, response_format=None, tools_to_call_from=None, **kwargs):
        task = next(_text(m) for m in messages if m.role == MessageRole.USER).replace("New task:", "").strip()
        observations = [_text(m) for m in messages if m.role == MessageRole.TOOL_RESPONSE]
        nums = [float(x) for x in re.findall(r"-?\d+(?:\.\d+)?", task)]
        op = "multiply" if re.search(r"multipl|times|product", task, re.I) else "add" if re.search(r"\badd|plus|sum", task, re.I) else None

        if not observations:                                    # 1st step: pick a tool
            if op and len(nums) >= 2:
                return self._call("math", {"a": nums[0], "b": nums[1], "op": op})
            return self._call("kb_lookup", {"query": task})

        obs = observations[-1].split("Observation:", 1)[-1].strip()
        if op and len(nums) >= 2:                               # 2nd step: final answer
            sym = "+" if op == "add" else "×"
            answer = f"{nums[0]:g} {sym} {nums[1]:g} = {obs} (computed with the math tool)."
        elif "NO_MATCH" in obs:
            answer = ("I could not find evidence for this in the knowledge base, so I won't guess. "
                      f"Follow-up: could you rephrase with key terms (e.g. 'agent tools' or 'RAG citations')?")
        else:
            lines = [l for l in obs.splitlines() if l.startswith("[kb:")][:2]
            parts = [re.sub(r"^(\[kb:\d+\]) \([^)]*\) (.*)$", r"\2 \1", l) for l in lines]
            answer = " ".join(parts)
        return self._call("final_answer", {"answer": answer})


USE_TINY_GPT2 = False                     # True -> use TransformersModel("sshleifer/tiny-gpt2") as the agent model
model = TransformersModel(model_id="sshleifer/tiny-gpt2", max_new_tokens=60) if USE_TINY_GPT2 else StubToolModel()
print("Agent model:", type(model).__name__)

Agent model: StubToolModel


## 4. The agent

In [6]:
agent = ToolCallingAgent(tools=[kb_tool, math_tool], model=model, max_steps=4, verbosity_level=1)
print("Tools available to the agent:", list(agent.tools))

Tools available to the agent: ['kb_lookup', 'math', 'final_answer']


## 5. Run the 3 test questions and inspect tool calls + final answers

In [7]:
def run_and_inspect(question):
    print("=" * 90, "\nQUESTION:", question)
    final = agent.run(question)
    for step in agent.memory.steps:
        for tc in (getattr(step, "tool_calls", None) or []):
            print(f"  🛠️  step {step.step_number}: {tc.name}({tc.arguments})")
        if getattr(step, "observations", None):
            print(f"      ↳ observation: {str(step.observations)[:160]!r}")
    print("FINAL ANSWER:", final)
    return final

answers = {q: run_and_inspect(q) for q in ["Add 12 and 30.", "Multiply 7 by 6.", "What is an agentic AI loop?"]}

QUESTION: Add 12 and 30.


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Add 12 and 30.                                                                                                  │
│                                                                                                                 │
╰─ StubToolModel - None ──────────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'math' with arguments: {'a': 12.0, 'b': 30.0, 'op': 'add'}                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: 42

...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': 'An agentic AI loop repeats observe -> think/plan ->    │
│ act (call a tool) -> observe the result, until the goal is reached or a step limit stops it. [kb:1]'}           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: An agentic AI loop repeats observe -> think/plan -> act (call a tool) -> observe the result, until 
the goal is reached or a step limit stops it. |kb:1]

Final answer: An agentic AI loop repeats observe -> think/plan -> act (call a tool) -> observe the result, until 
the goal is reached or a step limit stops it. [kb:1]

[Step 2: Duration 0.00 seconds]

  🛠️  step 1: kb_lookup({'query': 'What is an agentic AI loop?'})
      ↳ observation: '[kb:1] (course-notes/agents) An agentic AI loop repeats observe -> think/plan -> act (call a tool) -> observe the result, until the goal is reached or a step li'
  🛠️  step 2: final_answer({'answer': 'An agentic AI loop repeats observe -> think/plan -> act (call a tool) -> observe the result, until the goal is reached or a step limit stops it. [kb:1]'})
      ↳ observation: 'An agentic AI loop repeats observe -> think/plan -> act (call a tool) -> observe the result, until the goal is reached or a step limit stops it. [kb:1]'
FINAL ANSWER: An agentic AI loop repeats observe -> think/plan -> act (call a tool) -> observe the result, until the goal is reached or a step limit stops it. [kb:1]


In [8]:
# Extra check: a question the KB does not cover -> the agent says so and proposes a follow-up
_ = run_and_inspect("What is the capital of Mongolia?")

QUESTION: What is the capital of Mongolia?


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ What is the capital of Mongolia?                                                                                │
│                                                                                                                 │
╰─ StubToolModel - None ──────────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'kb_lookup' with arguments: {'query': 'What is the capital of Mongolia?'}                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: NO_MATCH

...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': "I could not find evidence for this in the knowledge    │
│ base, so I won't guess. Follow-up: could you rephrase with key terms (e.g. 'agent tools' or 'RAG citations')?"} │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: I could not find evidence for this in the knowledge base, so I won't guess. Follow-up: could you 
rephrase with key terms (e.g. 'agent tools' or 'RAG citations')?

Final answer: I could not find evidence for this in the knowledge base, so I won't guess. Follow-up: could you 
rephrase with key terms (e.g. 'agent tools' or 'RAG citations')?

[Step 2: Duration 0.00 seconds]

  🛠️  step 1: kb_lookup({'query': 'What is the capital of Mongolia?'})
      ↳ observation: 'NO_MATCH'
  🛠️  step 2: final_answer({'answer': "I could not find evidence for this in the knowledge base, so I won't guess. Follow-up: could you rephrase with key terms (e.g. 'agent tools' or 'RAG citations')?"})
      ↳ observation: "I could not find evidence for this in the knowledge base, so I won't guess. Follow-up: could you rephrase with key terms (e.g. 'agent tools' or 'RAG citations')"
FINAL ANSWER: I could not find evidence for this in the knowledge base, so I won't guess. Follow-up: could you rephrase with key terms (e.g. 'agent tools' or 'RAG citations')?


### What happens with `sshleifer/tiny-gpt2`?
We plug the exercise's tiny model into the **same** agent. It is a 2-layer test checkpoint with random-like weights: it emits gibberish, never valid tool-call JSON, so smolagents raises a parsing error at each step until `max_steps` is reached.

In [9]:
tiny = TransformersModel(model_id="sshleifer/tiny-gpt2", max_new_tokens=40)
tiny_agent = ToolCallingAgent(tools=[kb_tool, math_tool], model=tiny, max_steps=2, verbosity_level=0)
try:
    out = tiny_agent.run("Add 12 and 30.")
    print("Final answer:", repr(out)[:300])
except Exception as e:
    print("Agent error:", type(e).__name__, str(e)[:300])
for step in tiny_agent.memory.steps:
    err = getattr(step, "error", None)
    raw = getattr(getattr(step, "model_output_message", None), "content", None)
    if err or raw:
        print(f"step {getattr(step, 'step_number', '?')}: raw model output={str(raw)[:120]!r}\n   error={str(err)[:200]}")

Error while generating output:
Cannot use chat template functions because tokenizer.chat_template is not set and no template argument was passed! 
For information about writing templates and setting the tokenizer.chat_template attribute, please see the 
documentation at https://huggingface.co/docs/transformers/main/en/chat_templating

Agent error: AgentGenerationError Error while generating output:
Cannot use chat template functions because tokenizer.chat_template is not set and no template argument was passed! For information about writing templates and setting the tokenizer.chat_template attribute, please see the documentation at https://huggingface.co/docs/tra


### Optional: a real (small) instruction-tuned model
`TransformersModel("Qwen/Qwen2.5-1.5B-Instruct")` runs locally on CPU (~3 GB download) and **can** produce real tool calls through its chat template. Set `TRY_QWEN = True` to try it.

In [10]:
TRY_QWEN = True
if TRY_QWEN:
    qwen = TransformersModel(model_id="Qwen/Qwen2.5-1.5B-Instruct", max_new_tokens=256, device_map="cpu")
    qwen_agent = ToolCallingAgent(tools=[kb_tool, math_tool], model=qwen, max_steps=4, verbosity_level=0)
    for q in ["Add 12 and 30.", "What is an agentic AI loop?"]:
        try:
            out = qwen_agent.run(q)
        except Exception as e:
            out = f"(agent error: {type(e).__name__}: {str(e)[:150]})"
        calls = [(tc.name, tc.arguments) for s in qwen_agent.memory.steps for tc in (getattr(s, "tool_calls", None) or [])]
        print(f"Q: {q}\n  tool calls: {calls}\n  final: {out}\n")

Error while parsing tool call from model output: The model output does not contain any JSON blob.

Reached max steps.

Q: Add 12 and 30.
  tool calls: [('math', {'a': 12, 'b': 30, 'op': 'add'}), ('math', {'a': 12, 'b': 30, 'op': 'multiply'}), ('math', {'a': 12, 'b': 30, 'op': 'add'})]
  final: The sum of 12 and 30 is 42.



Q: What is an agentic AI loop?
  tool calls: [('kb_lookup', {'query': 'agentic AI loop'}), ('final_answer', {'answer': 'An agentic AI loop is a repeating cycle where an AI system observes its environment, plans actions based on its observations, performs those actions, then observes the results, and continues this process until the goal is achieved or the maximum number of iterations is reached.'})]
  final: An agentic AI loop is a repeating cycle where an AI system observes its environment, plans actions based on its observations, performs those actions, then observes the results, and continues this process until the goal is achieved or the maximum number of iterations is reached.



## Results & analysis

| Question | Tool calls (stub model) | Final answer |
|---|---|---|
| *Add 12 and 30.* | `math(a=12, b=30, op='add')` → 42, then `final_answer` | 12 + 30 = 42 (computed with the math tool). |
| *Multiply 7 by 6.* | `math(a=7, b=6, op='multiply')` → 42, then `final_answer` | 7 × 6 = 42 (computed with the math tool). |
| *What is an agentic AI loop?* | `kb_lookup(query=...)` → `[kb:1]`, then `final_answer` | Observe → think/plan → act → observe, until the goal or a step limit… **[kb:1]** |
| *(extra) What is the capital of Mongolia?* | `kb_lookup` → `NO_MATCH`, then `final_answer` | No evidence in the KB, so no guess, plus a suggested follow-up. |

Every run follows the smolagents loop: **model → tool call → observation stored in `agent.memory` → model → `final_answer`**. The KB answer keeps its citation tag. When there is no evidence, the agent says so instead of hallucinating.

**`sshleifer/tiny-gpt2`:** the agent cannot even start. `ToolCallingAgent` formats the conversation with the model's **chat template**, and this tiny test checkpoint has none (`tokenizer.chat_template is not set`). Even with a template, a 2-layer randomly initialised model cannot emit valid tool-call JSON. It is useful only to test that a pipeline loads, which is why the stub is the default.

**Qwen2.5-1.5B-Instruct (real local model):**
- It calls the tools on its own: `math(12, 30, 'add')` and `kb_lookup('agentic AI loop')`, then `final_answer`. This shows what a real tool-calling model adds.
- Its behaviour also shows the usual small-model weaknesses:
  - **redundant calls:** add, then multiply, then add again before answering;
  - **a lost citation:** the KB answer is a paraphrase without `[kb:1]`.
- In practice you would add a system-prompt rule ("always end KB-based answers with the [kb:N] tags you used"), validate the final answer against the observations, or use a larger model. The deterministic stub guarantees these properties by construction, which is why it is useful for tests.
